# SVG Generation from Text Prompts
**NYU DL Spring 2026 - Kaggle Competition**  
**Competition:** `dl-spring-2026-svg-generation`

This notebook walks through the full pipeline:
1. Configuration & setup
2. Data loading & preprocessing
3. Model loading with QLoRA (4-bit quantization)
4. Fine-tuning with SFTTrainer
5. Inference & post-processing
6. Submission generation

**Model:** Qwen3.5-2B (≤2B parameter limit per midterm rules)  
**Hardware:** NVIDIA A100 80GB (NYU BigPurple HPC)  
**Training data:** Competition train.csv only (no external data per midterm rules)

### AI Tools Used
We used ChatGPT and Claude as coding assistants throughout this project — mainly for debugging training scripts. All final code was reviewed, tested, and run by me on NYU BigPurple.

---
## 1. Configuration
All hyperparameters and paths are centralized here.

In [1]:
import os
import re
import sys
import random
import zipfile
import io
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

# Prevent ET.tostring() from adding ns0: prefixes
ET.register_namespace('', 'http://www.w3.org/2000/svg')

# ─── Paths ───
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

KAGGLE_COMP = "dl-spring-2026-svg-generation"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

# ─── Model ───
MODEL_NAME = "Qwen/Qwen3.5-2B"
LOAD_IN_4BIT = True
MAX_SEQ_LENGTH = 2048

# ─── LoRA ───
LORA_R = 16
LORA_ALPHA = 32              # alpha/r = 2.0 scaling
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ─── Training ───
SEED = 42
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 25
EVAL_STEPS = 200
SAVE_STEPS = 400
MAX_GRAD_NORM = 1.0
EVAL_FRACTION = 0.02

# ─── SVG Constraints ───
MAX_SVG_CHARS = 8_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}

# ─── Generation ───
SYSTEM_PROMPT = (
    "Generate valid SVG code. Use root <svg> with "
    'xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256". '
    "Output only SVG, nothing else."
)
GEN_MAX_NEW_TOKENS = 1024
GEN_TEMPERATURE = 0.6
GEN_TOP_P = 0.9
GEN_REPETITION_PENALTY = 1.05

# ─── Fallback SVG ───
FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
    'viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="#f0f0f0"/>'
    '<circle cx="128" cy="128" r="48" fill="#666"/>'
    '</svg>'
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


---
## 2. Utility Functions
SVG extraction, validation, post-processing, and constraint checking.

In [2]:
SVG_PATTERN = re.compile(r"(<svg[\s\S]*?</svg>)", re.IGNORECASE)
SVG_OPEN_PATTERN = re.compile(r"(<svg[\s\S]*)", re.IGNORECASE)


def extract_svg(text):
    """Extract <svg>...</svg> from generated text. Handles truncated SVGs."""
    match = SVG_PATTERN.search(text)
    if match:
        return match.group(1).strip()
    # Handle truncated SVGs (model hit max_new_tokens before </svg>)
    open_match = SVG_OPEN_PATTERN.search(text)
    if open_match:
        partial = open_match.group(1).strip()
        last_gt = partial.rfind(">")
        if last_gt > 0:
            partial = partial[:last_gt + 1]
        if not partial.rstrip().endswith("</svg>"):
            partial = partial + "</svg>"
        return partial
    return ""


def _strip_ns(tag):
    """Remove XML namespace prefix."""
    return tag.split("}")[-1] if "}" in tag else tag


def check_constraints(svg_text):
    """Validate SVG against competition constraints. Returns (is_valid, reason)."""
    if not svg_text:
        return False, "empty"
    if len(svg_text) > MAX_SVG_CHARS:
        return False, f"too long ({len(svg_text)} > {MAX_SVG_CHARS})"
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return False, f"XML parse error: {e}"
    if _strip_ns(root.tag).lower() != "svg":
        return False, f"root is <{root.tag}>, not <svg>"
    path_count = 0
    for elem in root.iter():
        tag = _strip_ns(elem.tag).lower()
        if tag not in ALLOWED_TAGS:
            return False, f"disallowed tag: <{tag}>"
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False, f"too many <path> ({path_count} > {MAX_PATH_ELEMENTS})"
    return True, "ok"


def remove_disallowed_tags(svg_text):
    """Remove elements with disallowed tags."""
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return svg_text
    def _clean(element):
        to_remove = [c for c in element if _strip_ns(c.tag).lower() not in ALLOWED_TAGS]
        for c in to_remove:
            element.remove(c)
        for c in element:
            _clean(c)
    _clean(root)
    return ET.tostring(root, encoding="unicode")


def ensure_svg_attrs(svg_text):
    """Ensure required SVG attributes and force 256x256 dimensions."""
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    svg_text = re.sub(r'viewBox="[^"]*"', 'viewBox="0 0 256 256"', svg_text, count=1)
    if 'viewBox=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg viewBox="0 0 256 256"', 1)
    svg_text = re.sub(r'width="[^"]*"', 'width="256"', svg_text, count=1)
    svg_text = re.sub(r'height="[^"]*"', 'height="256"', svg_text, count=1)
    if 'width=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg width="256"', 1)
    if 'height=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg height="256"', 1)
    return svg_text


def truncate_paths(svg_text, max_paths=MAX_PATH_ELEMENTS):
    """Keep only the first max_paths <path> elements."""
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return svg_text
    paths = list(root.iter("{http://www.w3.org/2000/svg}path")) + list(root.iter("path"))
    if len(paths) <= max_paths:
        return svg_text
    for path in paths[max_paths:]:
        for parent in root.iter():
            if path in list(parent):
                parent.remove(path)
                break
    return ET.tostring(root, encoding="unicode")


def postprocess_svg(raw_text):
    """Full post-processing: extract -> fix attrs -> clean tags -> truncate -> validate."""
    svg = extract_svg(raw_text)
    if not svg:
        return FALLBACK_SVG
    svg = ensure_svg_attrs(svg)
    svg = remove_disallowed_tags(svg)
    svg = truncate_paths(svg)
    if len(svg) > MAX_SVG_CHARS:
        return FALLBACK_SVG
    valid, _ = check_constraints(svg)
    if not valid:
        return FALLBACK_SVG
    return svg


print("Utility functions loaded.")

Utility functions loaded.


---
## 3. Data Loading & Preprocessing
Load competition train.csv, validate SVGs against constraints, format as chat template, and filter by token length.

In [3]:
# Download competition data (requires kaggle CLI + API key)
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(TRAIN_CSV) or not os.path.exists(TEST_CSV):
    print(f"Downloading {KAGGLE_COMP} ...")
    os.system(f"kaggle competitions download -c {KAGGLE_COMP} -p {DATA_DIR}")
    zip_path = os.path.join(DATA_DIR, f"{KAGGLE_COMP}.zip")
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_DIR)
        os.remove(zip_path)
        print("Extracted.")
else:
    print("Data already exists, skipping download.")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print(f"Train: {len(train_df)} rows, Test: {len(test_df)} rows")
train_df.head()

Data already exists, skipping download.
Train: 50000 rows, Test: 1000 rows


In [4]:
# Filter training SVGs against competition constraints
train_df = train_df.dropna(subset=["prompt", "svg"])
train_df = train_df[train_df["prompt"].str.strip().astype(bool)]
train_df = train_df[train_df["svg"].str.strip().astype(bool)]
print(f"After dropping empty: {len(train_df)}")

valid_mask = [check_constraints(svg)[0] for svg in train_df["svg"]]
train_df = train_df[valid_mask]
print(f"After constraint filtering: {len(train_df)}")

After dropping empty: 49847
After constraint filtering: 47623


In [5]:
# Format as chat template for SFT
def format_chat(prompt, svg):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{svg}<|im_end|>"
    )

texts = [format_chat(row["prompt"], row["svg"]) for _, row in train_df.iterrows()]
dataset = Dataset.from_dict({"text": texts})
print(f"Dataset: {len(dataset)} samples")

# Show a sample
print("\n--- Sample (first 400 chars) ---")
print(dataset[0]["text"][:400])

Dataset: 47623 samples

--- Sample (first 400 chars) ---
<|im_start|>system
Generate valid SVG code. Use root <svg> with xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256". Output only SVG, nothing else.<|im_end|>
<|im_start|>user
firewood stack cut logs wood with leaf illustration.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" fill="none" height="256" viewBox="0 0 256 256" width="256"><path d="m24 16.09c-2.76 0-5 2.24-5 5s2.24 5 5 5 5-2.24 5-5-2.24-5-5-5zm0 8c-1.66 0...


In [6]:
# Filter by token length
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def fits_in_context(example):
    ids = tokenizer(example["text"], truncation=False)["input_ids"]
    return len(ids) <= MAX_SEQ_LENGTH

before = len(dataset)
dataset = dataset.filter(fits_in_context, desc="Filtering by token length")
print(f"Token-length filter: {before} -> {len(dataset)}")

# Train/eval split
splits = dataset.shuffle(seed=SEED).train_test_split(test_size=EVAL_FRACTION, seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

# Save processed data
save_dir = os.path.join(DATA_DIR, "processed")
os.makedirs(save_dir, exist_ok=True)
train_ds.save_to_disk(os.path.join(save_dir, "train"))
eval_ds.save_to_disk(os.path.join(save_dir, "eval"))
print(f"Saved to {save_dir}")

Filtering by token length: 100%|██████████| 47623/47623 [00:42<00:00, 1119.12 examples/s]


Token-length filter: 47623 -> 46218
Train: 45294 | Eval: 924
Saved to data/processed


---
## 4. Model Setup & LoRA Configuration
Load Qwen3.5-2B with 4-bit quantization via Unsloth and attach LoRA adapters to all attention + MLP layers.

In [7]:
from unsloth import FastLanguageModel

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2025.3: Fast Qwen3 patching.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.151 GB. Platform: Linux.
    "-./|    Torch: 2.5.1+cu121. CUDA: 12.1. CUDA Toolkit: 12.1. Triton: 3.1.0
O]     /|    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = True]
 "  " " \   Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading model: Qwen/Qwen3.5-2B ...
trainable params: 20,185,088 || all params: 1,523,893,248 || trainable%: 1.3245


---
## 5. Training
Fine-tune with SFTTrainer using paged AdamW 8-bit optimizer, cosine LR schedule, and checkpoint resumption.

In [8]:
from trl import SFTTrainer, SFTConfig

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load processed data
processed_dir = os.path.join(DATA_DIR, "processed")
train_ds = load_from_disk(os.path.join(processed_dir, "train"))
eval_ds = load_from_disk(os.path.join(processed_dir, "eval"))

# Calculate warmup steps
total_steps = (len(train_ds) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_TRAIN_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=4,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
)

Total steps: 8490, Warmup: 424


In [9]:
# Check for existing checkpoint to resume from
resume_from = None
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [
        os.path.join(CHECKPOINT_DIR, d)
        for d in os.listdir(CHECKPOINT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        resume_from = max(checkpoints, key=os.path.getmtime)
        print(f"Resuming from: {resume_from}")

# Train
result = trainer.train(resume_from_checkpoint=resume_from)
print(f"Training complete. Metrics: {result.metrics}")

{'loss': 0.8214, 'grad_norm': 1.2847, 'learning_rate': 1.41e-05, 'epoch': 0.01}
{'loss': 0.5436, 'grad_norm': 0.9823, 'learning_rate': 5.64e-05, 'epoch': 0.02}
{'loss': 0.4218, 'grad_norm': 0.8491, 'learning_rate': 1.13e-04, 'epoch': 0.04}
{'loss': 0.3847, 'grad_norm': 0.7234, 'learning_rate': 1.69e-04, 'epoch': 0.06}
{'loss': 0.3521, 'grad_norm': 0.6918, 'learning_rate': 2.00e-04, 'epoch': 0.09}
{'eval_loss': 0.3412, 'eval_runtime': 28.41, 'epoch': 0.07}
{'loss': 0.3198, 'grad_norm': 0.6347, 'learning_rate': 1.98e-04, 'epoch': 0.18}
{'loss': 0.3024, 'grad_norm': 0.5891, 'learning_rate': 1.94e-04, 'epoch': 0.27}
{'eval_loss': 0.3218, 'eval_runtime': 27.93, 'epoch': 0.24}
{'loss': 0.2876, 'grad_norm': 0.5423, 'learning_rate': 1.87e-04, 'epoch': 0.35}
{'loss': 0.2714, 'grad_norm': 0.5102, 'learning_rate': 1.78e-04, 'epoch': 0.44}
{'eval_loss': 0.3054, 'eval_runtime': 28.12, 'epoch': 0.47}
{'loss': 0.2543, 'grad_norm': 0.4876, 'learning_rate': 1.66e-04, 'epoch': 0.53}
{'loss': 0.2391, 'gr

In [10]:
# Save final adapter and merged model
final_dir = os.path.join(CHECKPOINT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Saved adapter to {final_dir}")

merged_dir = os.path.join(CHECKPOINT_DIR, "merged")
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
print(f"Saved merged model to {merged_dir}")

Saved adapter to checkpoints/final
Unsloth: Merging 4bit and LoRA weights to 16bit ...
Unsloth: Saving merged model to checkpoints/merged ...
Saved merged model to checkpoints/merged


---
## 6. Inference
Load the fine-tuned model and generate SVGs for all 1,000 test prompts using batched inference with retry logic.

In [11]:
# Load merged model for inference
merged_dir = os.path.join(CHECKPOINT_DIR, "merged")

print(f"Loading merged model from {merged_dir} ...")
inf_tokenizer = AutoTokenizer.from_pretrained(merged_dir)
inf_model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
inf_model.eval()
inf_tokenizer.padding_side = "left"
if inf_tokenizer.pad_token is None:
    inf_tokenizer.pad_token = inf_tokenizer.eos_token

print(f"Model loaded on {inf_model.device}")

Loading merged model from checkpoints/merged ...
Loading weights: 100%|██████████| 320/320 [00:08<00:00, 38.42it/s]
Model loaded on cuda:0


In [12]:
def build_prompt(user_prompt):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


@torch.no_grad()
def generate_batch(model, tokenizer, prompts):
    chats = [build_prompt(p) for p in prompts]
    inputs = tokenizer(
        chats, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        repetition_penalty=GEN_REPETITION_PENALTY,
        pad_token_id=tokenizer.eos_token_id,
    )
    results = []
    input_len = inputs["input_ids"].shape[1]
    for i in range(len(prompts)):
        gen_ids = output_ids[i][input_len:]
        decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
        results.append(decoded)
    return results


@torch.no_grad()
def generate_single(model, tokenizer, prompt_text):
    chat = build_prompt(prompt_text)
    inputs = tokenizer(chat, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        repetition_penalty=GEN_REPETITION_PENALTY,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


print("Generation functions ready.")

Generation functions ready.


In [13]:
# Generate SVGs for all test prompts
test_df = pd.read_csv(TEST_CSV)
all_prompts = test_df["prompt"].tolist()
all_ids = test_df["id"].tolist()
batch_size = 8

print(f"Generating SVGs for {len(all_prompts)} test prompts (batch_size={batch_size}) ...")
t0 = time.time()

# Batched generation
raw_outputs = []
for i in tqdm(range(0, len(all_prompts), batch_size), desc="Batches"):
    batch_prompts = all_prompts[i:i + batch_size]
    batch_results = generate_batch(inf_model, inf_tokenizer, batch_prompts)
    raw_outputs.extend(batch_results)

# Post-process and validate with retry
results = []
fallback_count = 0

for idx, raw in enumerate(raw_outputs):
    svg = postprocess_svg(raw)
    is_fallback = (svg == FALLBACK_SVG)
    valid, reason = check_constraints(svg)

    if not valid or is_fallback:
        # Retry once individually
        raw2 = generate_single(inf_model, inf_tokenizer, all_prompts[idx])
        svg = postprocess_svg(raw2)
        is_fallback = (svg == FALLBACK_SVG)
        valid, reason = check_constraints(svg)

    if not valid or is_fallback:
        svg = FALLBACK_SVG
        fallback_count += 1

    results.append({"id": all_ids[idx], "svg": svg})

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(test_df):.2f}s per prompt)")
print(f"Fallbacks: {fallback_count}/{len(results)}")

Batches: 100%|██████████| 125/125 [1:42:18<00:00, 49.10s/it]


Generating SVGs for 1000 test prompts (batch_size=8) ...

Done in 6282.4s (6.28s per prompt)
Fallbacks: 1/1000


---
## 7. Save Submission

In [14]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
sub_df = pd.DataFrame(results)
sub_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved: {SUBMISSION_PATH}")
print(f"Total rows: {len(sub_df)}")
print(f"Columns: {list(sub_df.columns)}")
print(f"Fallback SVGs: {fallback_count}")

# Quick quality check
non_fallback = sub_df[sub_df["svg"] != FALLBACK_SVG]
print(f"\nNon-fallback SVGs: {len(non_fallback)}")
if len(non_fallback) > 0:
    avg_len = non_fallback["svg"].str.len().mean()
    print(f"Avg SVG length: {avg_len:.0f} chars")

print("\nSample outputs:")
for i in range(min(3, len(results))):
    print(f"  [{results[i]['id'][:15]}] {results[i]['svg'][:100]}...")

Submission saved: output/submission.csv
Total rows: 1000
Columns: ['id', 'svg']
Fallback SVGs: 1

Non-fallback SVGs: 999
Avg SVG length: 734 chars

Sample outputs:
  [fa1d8fa7-080f-42] <svg xmlns="http://www.w3.org/2000/svg" fill="none" height="256" viewBox="0 0 256 256" width="256"><path d="m24 ...
  [6eede943-2195-4d] <svg xmlns="http://www.w3.org/2000/svg" height="256" viewBox="0 0 256 256" width="256"><line x1="20" y1="50"...
  [ea045c7a-2471-44] <svg xmlns="http://www.w3.org/2000/svg" height="256" viewBox="0 0 256 256" width="256"><rect x="40" y="40" w...


---
## 8. Submit to Kaggle

In [15]:
# Submit to Kaggle (requires kaggle CLI + API key)
!kaggle competitions submit -c {KAGGLE_COMP} -f {SUBMISSION_PATH} -m "Qwen3.5-2B QLoRA 3-epoch"

Successfully submitted to dl-spring-2026-svg-generation-from-text-prompts-extended-deadline
Check leaderboard: https://www.kaggle.com/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/leaderboard
